## knn-v3.ipynb

Extends the v2 model by adding three contributor-derived feature blocks
produced by `weights-contributors.ipynb`:

| New block | Columns | What it captures |
|---|---|---|
| `album_role_family_matrix.npz` | 7 | Normalised distribution of contribution types (performance / production / technical / writing / visual_packaging / business_label / other) |
| `album_instrument_matrix.npz` | ~984 (post-filter) | Normalised instrument presence profile |
| `album_contributor_counts_matrix.npz` | 7 | Min-max scaled distinct contributor counts per role family |

All other pipeline steps (universe expansion, safe column pruning, L2
normalisation, brute-force cosine search, model serialisation) are
identical to `knn-v2.ipynb`.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

os.makedirs('../data/features',   exist_ok=True)
os.makedirs('../data/model_v3',   exist_ok=True)

In [2]:
# Load row index
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# v2 baseline blocks
X_tags        = load_npz('../data/features/album_tags_matrix.npz')
X_labels      = load_npz('../data/features/album_labels_matrix.npz')
X_types       = load_npz('../data/features/album_types_matrix.npz')
X_ratings     = load_npz('../data/features/album_ratings_matrix.npz')
X_country     = load_npz('../data/features/album_country_matrix.npz')
X_track_stats = load_npz('../data/features/album_track_stats_matrix.npz')

# v3 new blocks
X_role_family  = load_npz('../data/features/album_role_family_matrix.npz')
X_instrument   = load_npz('../data/features/album_instrument_matrix.npz')
X_contrib_cnt  = load_npz('../data/features/album_contributor_counts_matrix.npz')

print('Feature blocks loaded:')
for name, X in [
    ('X_tags',        X_tags),
    ('X_labels',      X_labels),
    ('X_types',       X_types),
    ('X_ratings',     X_ratings),
    ('X_country',     X_country),
    ('X_track_stats', X_track_stats),
    ('X_role_family', X_role_family),
    ('X_instrument',  X_instrument),
    ('X_contrib_cnt', X_contrib_cnt),
]:
    print(f'  {name:<18} {str(X.shape):<25} nnz={X.nnz:,}')

Feature blocks loaded:
  X_tags             (1008102, 3041)           nnz=3,004,997
  X_labels           (1008102, 3469)           nnz=402,047
  X_types            (1008102, 10)             nnz=402,047
  X_ratings          (1008102, 1)              nnz=44,334
  X_country          (1008102, 2263)           nnz=883,503
  X_track_stats      (1008102, 12)             nnz=11,139,577
  X_role_family      (1008102, 7)              nnz=1,155,920
  X_instrument       (1008102, 591)            nnz=1,459,235
  X_contrib_cnt      (1008102, 7)              nnz=1,155,920


In [3]:
# Expand all matrices to full album universe
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

all_blocks = {
    'X_tags':        X_tags,
    'X_labels':      X_labels,
    'X_types':       X_types,
    'X_ratings':     X_ratings,
    'X_country':     X_country,
    'X_track_stats': X_track_stats,
    'X_role_family': X_role_family,
    'X_instrument':  X_instrument,
    'X_contrib_cnt': X_contrib_cnt,
}

if len(album_id_order) < len(full_album_ids):
    print(f'Expanding {len(album_id_order):,} → {len(full_album_ids):,} albums...')
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    all_blocks = {k: _expand(v, current_pos, n_full) for k, v in all_blocks.items()}
    album_id_order = full_album_ids.tolist()

X_tags, X_labels, X_types, X_ratings, X_country, X_track_stats, \
    X_role_family, X_instrument, X_contrib_cnt = all_blocks.values()

Expanding 1,008,102 → 2,241,402 albums...


In [4]:
from scipy.sparse import hstack  # re-import in case cell is run standalone

# --- Block weights ---------------------------------------------------
# Ratios between weights are what matter after L2 normalisation.
# Contributor blocks (role_family, instrument, counts) cover only a
# fraction of albums (~525K–707K vs 2.2M), so start them at 1.0 and
# dial back if they dominate results for well-covered albums.
# Country remains at 0.2 from v2 tuning.
W_TAGS         = 1.0
W_LABELS       = 1.0
W_TYPES        = 1.0
W_RATINGS      = 1.0
W_COUNTRY      = 0.2   # tuned in v2 — adjust if needed
W_TRACK_STATS  = 1.0
W_ROLE_FAMILY  = 1.0   # <-- tune: try 0.3–1.0
W_INSTRUMENT   = 1.0   # <-- tune: try 0.5–1.5
W_CONTRIB_CNT  = 0.5   # scale contribution — counts are a secondary signal

X_final_v3 = hstack([
    X_tags         * W_TAGS,
    X_labels       * W_LABELS,
    X_types        * W_TYPES,
    X_ratings      * W_RATINGS,
    X_country      * W_COUNTRY,
    X_track_stats  * W_TRACK_STATS,
    X_role_family  * W_ROLE_FAMILY,
    X_instrument   * W_INSTRUMENT,
    X_contrib_cnt  * W_CONTRIB_CNT,
]).tocsr()

print(f'X_final_v3: {X_final_v3.shape[0]:,} albums x {X_final_v3.shape[1]:,} features  (nnz={X_final_v3.nnz:,})')
print(f'\nBlock summary (with weights):')
for name, X, w in [
    ('tags',           X_tags,        W_TAGS),
    ('labels',         X_labels,      W_LABELS),
    ('types',          X_types,       W_TYPES),
    ('ratings',        X_ratings,     W_RATINGS),
    ('country',        X_country,     W_COUNTRY),
    ('track_stats',    X_track_stats, W_TRACK_STATS),
    ('role_family',    X_role_family, W_ROLE_FAMILY),
    ('instrument',     X_instrument,  W_INSTRUMENT),
    ('contrib_counts', X_contrib_cnt, W_CONTRIB_CNT),
]:
    print(f'  {name:<18} w={w}   cols={X.shape[1]:,}')

X_final_v3: 2,241,402 albums x 9,401 features  (nnz=19,647,580)

Block summary (with weights):
  tags               w=1.0   cols=3,041
  labels             w=1.0   cols=3,469
  types              w=1.0   cols=10
  ratings            w=1.0   cols=1
  country            w=0.2   cols=2,263
  track_stats        w=1.0   cols=12
  role_family        w=1.0   cols=7
  instrument         w=1.0   cols=591
  contrib_counts     w=0.5   cols=7


In [5]:
# Safe column pruning — identical to knn.ipynb / knn-v2.ipynb
col_nnz      = np.diff(X_final_v3.tocsc().indptr)
row_lengths  = np.diff(X_final_v3.indptr)
has_features = row_lengths > 0

col_nnz_vals           = col_nnz[X_final_v3.indices]
nonempty_rows          = np.where(has_features)[0]
row_starts             = X_final_v3.indptr[nonempty_rows]
max_col_nnz_per_album  = np.zeros(X_final_v3.shape[0], dtype=col_nnz.dtype)
max_col_nnz_per_album[nonempty_rows] = np.maximum.reduceat(col_nnz_vals, row_starts)

safe_threshold = int(max_col_nnz_per_album[has_features].min())
keep_cols      = col_nnz >= safe_threshold
X_knn_v3       = X_final_v3[:, keep_cols]

print(f'Safe threshold : {safe_threshold}')
print(f'Columns before : {X_final_v3.shape[1]:,}')
print(f'Columns after  : {X_knn_v3.shape[1]:,}  ({keep_cols.mean()*100:.1f}% retained)')
print(f'Albums with features: {has_features.sum():,}  ({has_features.mean()*100:.1f}%)')

Safe threshold : 10
Columns before : 9,401
Columns after  : 6,459  (68.7% retained)
Albums with features: 1,008,102  (45.0%)


In [6]:
# Subset to annotated albums and L2-normalise
X_knn_annotated_v3     = X_knn_v3[has_features].copy()
album_ids_annotated_v3 = np.array(album_id_order)[has_features]

nan_count = np.isnan(X_knn_annotated_v3.data).sum()
if nan_count:
    print(f'Removing {nan_count:,} NaN entries...')
    np.nan_to_num(X_knn_annotated_v3.data, nan=0.0, copy=False)
    X_knn_annotated_v3.eliminate_zeros()

X_knn_norm_v3 = normalize(X_knn_annotated_v3, norm='l2')
print(f'Fitting on {X_knn_norm_v3.shape[0]:,} albums x {X_knn_norm_v3.shape[1]:,} features')

Removing 7,657 NaN entries...
Fitting on 1,008,102 albums x 6,459 features


In [7]:
model_v3 = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model_v3.fit(X_knn_norm_v3)
print('Model v3 fitted.')

Model v3 fitted.


In [8]:
# Sanity check
distances, indices = model_v3.kneighbors(X_knn_norm_v3[0], n_neighbors=11)

print(f'Query album id: {album_ids_annotated_v3[0]}')
print(f"\n{'rank':<6} {'album_id':<40} {'cosine distance':>15}")
print('-' * 62)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = '(query)' if rank == 0 else ''
    print(f"{rank:<6} {str(album_ids_annotated_v3[idx]):<40} {dist:>15.4f}  {label}")

Query album id: 4

rank   album_id                                 cosine distance
--------------------------------------------------------------
0      4                                                 0.0000  (query)
1      131486                                            0.0969  
2      576                                               0.1056  
3      126314                                            0.1165  
4      27794                                             0.1179  
5      36458                                             0.1245  
6      58306                                             0.1246  
7      32587                                             0.1256  
8      70475                                             0.1258  
9      79059                                             0.1265  
10     25343                                             0.1269  


In [9]:
joblib.dump(model_v3,                   '../data/model_v3/knn_model_v3.joblib')
save_npz('../data/model_v3/X_knn_norm_v3.npz', X_knn_norm_v3)
np.save('../data/model_v3/album_ids_annotated_v3.npy', album_ids_annotated_v3)
np.save('../data/model_v3/has_features_v3.npy',        has_features)

print('Saved:')
print('  ../data/model_v3/knn_model_v3.joblib')
print('  ../data/model_v3/X_knn_norm_v3.npz')
print('  ../data/model_v3/album_ids_annotated_v3.npy')
print('  ../data/model_v3/has_features_v3.npy')

Saved:
  ../data/model_v3/knn_model_v3.joblib
  ../data/model_v3/X_knn_norm_v3.npz
  ../data/model_v3/album_ids_annotated_v3.npy
  ../data/model_v3/has_features_v3.npy
